# 부록: 골든셋과 Jev 판사로 챗봇 프롬프트 자동 개선하기

챗봇이 틀린 답을 하면 보통 프롬프트를 고칩니다. 이 부록은 그 작업을 손 대신 평가로 진행합니다. 가상 헬스장 **핏플러스**의 안내 챗봇은 54문장짜리 안내 문서에서 검색기로 문장을 찾아 답하는데, 문서에 없는 질문에는 그럴듯한 답을 지어냅니다. 이 문제를 이렇게 고칩니다.

1. 안내 문서를 검색해 답하는 **RAG 챗봇**을 만들고, 문서 밖 내용을 지어내는 문제를 확인합니다(2절).
2. 챗봇이 잘 답해야 할 질문을 **골든셋**으로 만듭니다(3절).
3. **Jev** 판사를 만들고, 골든셋으로 판사가 믿을 만한지 점검합니다(4절).
4. DeepEval **Prompt Optimization**으로 프롬프트를 고치고, 고치는 과정과 고치기 전후를 확인합니다(5절).

**학습 목표**

- `Synthesizer`로 문서에서 골든을 만들고, 합성이 못 만드는 질문을 보완할 수 있다.
- 평가 기준을 참·거짓 문장으로 바꿔 `JevEval` 판사를 만들고, 골든셋으로 점검할 수 있다.
- 골든셋을 최적화용과 확인용으로 나누고, `PromptOptimizer`로 프롬프트를 고쳐 전후를 비교할 수 있다.

공식 문서: [JevEval](https://deepeval.com/docs/metrics-jev-eval) · [Prompt Optimization](https://deepeval.com/docs/prompt-optimization-introduction) · [골든](https://deepeval.com/docs/evaluation-datasets) · [Synthesizer](https://deepeval.com/docs/golden-synthesizer)

## 1. 실습 준비

`.env`에 Jev 키 `TYPESAFE_API_KEY`와 OpenAI 키 `OPENAI_API_KEY`를 넣어 둡니다. 키 이름은 `.env.example`에 있습니다.

LLM은 두 가지를 씁니다. 만드는 일(챗봇 답변, 골든 생성)은 생성 모델 `gpt-4.1`, 거르고 고치는 일(골든 품질 검사, 프롬프트 수정)은 판단 모델 `gpt-5.6-luna`가 맡습니다. 답변 채점은 Jev가 합니다.

In [ ]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

# deepeval을 불러오기 전에 설정합니다.
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"  # 사용 기록을 외부로 보내지 않음
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "30"  # 응답 없이 멈춘 호출을 30초 뒤 다시 보냄(기본 약 88초)
os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "4"  # 첫 호출을 포함한 시도 횟수 상한(기본 2회)
import deepeval

# material_dir는 data가 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data"

load_dotenv(find_dotenv(usecwd=True))  # 단원 폴더의 .env를 먼저 읽고
load_dotenv(material_dir.resolve().parent / ".env")  # 없는 키만 상위 실습자료 .env에서 채웁니다.

# 이 부록에서 쓰는 LLM: 만드는 일은 생성 모델, 거르고 고치는 일은 판단 모델이 맡습니다.
generation_model_name = "gpt-4.1"  # 생성 모델: 챗봇 답변, 골든(질문과 정답 답변) 생성
judgment_model_name = "gpt-5.6-luna"  # 판단 모델: 골든 품질 검사, 프롬프트 진단과 수정

# 판정 결과가 모델 버전에 따라 달라지지 않게 Jev 버전을 고정합니다. 답변 채점은 Jev가 합니다.
jev_model_name = os.getenv("TYPESAFE_MODEL", "jev-1.13.0")

print("DeepEval:", deepeval.__version__)
print("생성 모델:", generation_model_name, "/ 판단 모델:", judgment_model_name)
print("Jev 모델:", jev_model_name)
print("Jev 키 읽음:", os.getenv("TYPESAFE_API_KEY") is not None)

## 2. RAG 챗봇 만들기

### 2-1. 안내 문서 읽기

핏플러스 안내 문서(수업용 가상 자료)는 `data/부록/fitplus_guide.md`에 있습니다. `## 유형` 제목 아래에 `- 문장` 목록으로, 9개 유형 54문장입니다.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", None)  # 긴 문장을 자르지 않습니다.

# 안내 문서(MD) 읽기
guide_md = (data_dir / "부록" / "fitplus_guide.md").read_text(encoding="utf-8")

# "## 유형" 줄은 유형 이름으로, "- 문장" 줄은 안내 문장으로 읽습니다.
guide_rows = []
section = ""
for line in guide_md.splitlines():
    if line.startswith("## "):
        section = line[3:]
    elif line.startswith("- "):
        guide_rows.append({"유형": section, "문장": line[2:]})

# 문장 표, 그리고 판사 근거로 쓸 문장 목록
guide_table = pd.DataFrame(guide_rows)
guide_sentences = list(guide_table["문장"])

# 유형별 문장 수 확인
print("안내 문장", len(guide_sentences), "개")
display(guide_table.groupby("유형", sort=False).size().rename("문장 수").to_frame())

### 2-2. 저장된 Chroma로 검색기 만들기

실제 안내 챗봇처럼, 질문이 들어오면 검색기로 관련 문장만 찾아 답하게 합니다. 안내 문장 54개를 하나씩 `Document`로 만들어 넣은 Chroma 벡터 저장소를 `data/부록/chroma_fitplus`에 미리 저장해 두었습니다. 문장을 다시 임베딩하지 않도록 이 저장소를 불러와, 질문과 의미가 가까운 문장 4개를 찾는 dense 검색기를 만듭니다. 임베딩 설정은 day59와 같습니다.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 질문을 임베딩할 모델: 저장소를 만들 때와 같은 설정이어야 합니다(day59와 같은 설정).
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 입력 길이를 미리 검사해 나누는 자동 처리를 끕니다
)

# 미리 저장해 둔 Chroma 저장소 불러오기(안내 문장 하나가 Document 하나, 유형은 메타데이터)
vector_store = Chroma(
    collection_name="fitplus_guide",
    embedding_function=embedding_model,
    persist_directory=str(data_dir / "부록" / "chroma_fitplus"),
)
print("저장된 문장:", len(vector_store.get()["ids"]), "개")

# 질문과 의미가 가까운 문장 4개를 찾는 dense 검색기
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# 검색기 확인: 질문 하나로 찾은 문장 4개
for doc in retriever.invoke("주말에도 문 열어요?"):
    print(doc.metadata["유형"], "/", doc.page_content)

### 2-3. RAG 답변 체인과 문제 확인

교안 02처럼 시스템 프롬프트(챗봇의 규칙)와 휴먼 프롬프트(검색된 문장과 질문 자리)로 나눕니다. 질문이 들어오면 검색기가 찾은 문장 4개가 `{context}`에, 질문이 `{input}`에 들어갑니다. 시스템 프롬프트 둘째 문장은 일부러 넣은 실수입니다.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# 챗봇 답변 모델(1절의 생성 모델)
answer_llm = ChatOpenAI(model=generation_model_name, use_responses_api=True)


def format_docs(docs):
    # 검색된 문장을 한 줄에 하나씩 이어 붙입니다.
    lines = []
    for doc in docs:
        lines.append(doc.page_content)
    return "\n".join(lines)


# 시스템 프롬프트: 챗봇의 역할과 답하는 규칙. 둘째 문장이 일부러 넣은 실수입니다.
initial_system_prompt = (
    "너는 헬스장 '핏플러스'의 회원 안내 챗봇이야. 친절하고 짧게 답해.\n"
    "회원이 실망하지 않게, 안내 문서에 없는 내용을 물으면 다른 헬스장에서 흔한 내용으로 구체적으로 답해."
)
# 프롬프트 템플릿: 시스템(규칙) + 휴먼(검색된 문장과 질문)
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", initial_system_prompt),
    ("human", "[안내 문서]\n{context}\n\n[회원 질문]\n{input}"),
])

# 질문 -> 검색기로 문장 4개 찾기 -> 프롬프트 -> 답변 모델 -> 문자열
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | answer_prompt
    | answer_llm
    | StrOutputParser()
)

# 문서에 없는 질문 하나에 처음 프롬프트로 답해 보기
reply = rag_chain.invoke("PT 체험 수업은 무료예요?")
print(reply)

챗봇은 문서에 PT 체험 이야기가 없다고 말하면서도, "보통 1회 몇만 원 정도"처럼 다른 헬스장 기준의 내용을 덧붙입니다. 답변 문장은 실행마다 다르지만, 여러 번 실행해도 이렇게 문서 밖 정보를 섞는 모습은 매번 나왔습니다. 이 실수를 고치는 것이 목표입니다.

## 3. 골든셋 만들기

### 3-1. 골든은 아직 풀지 않은 시험 문제입니다

**골든**은 질문(`input`), 정답 답변(`expected_output`), 정답 근거(`context`: 이 질문에 답하는 데 필요한 문서 부분)를 묶은 것이고, 골든을 모은 것이 **골든셋**입니다.

골든은 사람이 직접 쓰거나, DeepEval의 `Synthesizer`로 문서 파일에서 만들 수 있습니다. 3-2에서는 Synthesizer로 만들고, 3-3에서 사람이 보완합니다.

### 3-2. Synthesizer로 문서 파일에서 골든 만들기

DeepEval의 `Synthesizer`는 문서 파일을 직접 읽어 골든을 만듭니다(`a_generate_goldens_from_docs`). 주피터에서는 비동기 버전(`a_` 붙은 함수)을 `await`로 부릅니다. 동기 버전(`generate_goldens_from_docs`)은 주피터 커널의 실행 루프와 충돌해 셀이 끝나지 않을 수 있습니다. 안에서는 세 단계가 돕니다.

1. **문서 자르기:** 파일(`.md`, `.txt`, `.pdf`, `.docx`)을 `chunk_size` 토큰 단위 청크(chunk)로 자릅니다.
2. **근거 묶음 만들기:** 청크를 무작위로 `max_contexts_per_document` × `max_retries`개 뽑아, 품질 검사 LLM(`critic_model`)이 명확성·깊이·구조·관련성 평균으로 한꺼번에 채점합니다. 그중 점수가 높은 청크를 근거 묶음 수만큼 고릅니다. 이 부록은 `max_retries=1`이라 9개를 뽑아 9개를 모두 쓰므로, 점수만 매기고 거르지는 않습니다. 기본값 3이면 27개 중 상위 9개를 골라 더 나은 청크가 남지만 채점 호출이 세 배가 됩니다.
    - `max_context_length`가 2 이상이면, 임베딩(`embedder`)으로 그 청크와 가장 비슷한 청크를 최대 `max_context_length - 1`개 붙여 근거 묶음(context)을 만듭니다. 여러 청크를 함께 봐야 답할 수 있는 질문을 만들기 위해서입니다. 문서 전체를 군집으로 나누는 방식이 아니라, 뽑은 청크 하나에 가까운 이웃 청크를 붙이는 방식입니다.
3. **골든 만들기:** 근거 묶음마다 그 내용으로 답할 수 있는 질문과 정답 답변을 LLM으로 만듭니다. 만든 질문은 품질 검사 LLM이 0~1점(1이 가장 명확하고 답할 수 있는 질문)으로 채점하고, 기준 점수(`synthetic_input_quality_threshold`, 기본값 0.5)보다 낮으면 고쳐 씁니다. 이 채점과 고쳐 쓰기를 `max_quality_retries`번까지 반복합니다.

설정의 뜻은 이렇습니다.

- `StylingConfig`: 어떤 상황의 질문을 어떤 말투와 형식으로 만들지 정합니다. 여기서는 한국어 회원 질문과 짧은 한국어 답변입니다.
- `EvolutionConfig(num_evolutions=0)`: 만든 질문을 **몇 번 다시 쓸지(진화 횟수)** 정합니다. 한 번 다시 쓸 때마다 진화 유형 7개(추론 넣기, 여러 근거 묶기, 구체화, 조건 붙이기, 비교, 가정, 범위 넓히기) 중 하나를 무작위로 골라 질문을 더 어렵게 고칩니다.
    - `0`: 다시 쓰지 않고 처음 만든 질문을 그대로 씁니다. 예: "PT 이용권은 언제까지 써야 하나요?"
    - `1`(기본값): 한 번 고칩니다. 예: "PT 이용권을 사고 7개월 뒤에 써도 되나요?"처럼 조건이 붙습니다.
    - `2` 이상: 고친 질문을 그 횟수만큼 더 고칩니다. 숫자가 클수록 질문이 길고 복잡해집니다.

    회원이 실제로 하는 단순한 질문을 원해서 0으로 뒀습니다.
- `ContextConstructionConfig`:
    - `chunk_size=100`: 청크 하나의 토큰 수(대략 안내 문장 한두 개)입니다. 청크 수는 따로 정하지 않고 이 값과 `chunk_overlap`으로 정해지며, 이 문서는 청크 36개로 나뉩니다.
    - `chunk_overlap=0`: 앞뒤 청크가 겹치는 토큰 수입니다. 0이면 겹치지 않으므로 경계에 걸린 문장은 두 청크로 잘립니다. 잘린 청크로 만든 골든은 정답 답변에 내용이 일부만 담기기도 합니다. 겹치게 두면(예: 50) 잘린 문장이 다음 청크에 이어서 들어가 이런 일이 줄어듭니다.
    - `max_contexts_per_document=9`: 그 청크들 중에서 질문을 만들 근거 묶음을 몇 개 고를지 정합니다. 청크 수가 아닙니다. 근거 묶음마다 골든을 `max_goldens_per_context`개 만들므로 9 × 1 = 골든 9개가 나옵니다. 자동으로 정해 주는 옵션은 없고, 보통 "원하는 골든 수 ÷ `max_goldens_per_context`"로 정합니다. 청크 수보다 크게 주면 청크 수로 줄여 모든 청크를 씁니다.
    - `max_context_length=1`: 근거 묶음 하나에 넣을 청크 수입니다. 1이면 이웃 청크를 붙이지 않고 청크 하나가 곧 근거 묶음이 되어, 청크 하나로 답할 수 있는 단순한 질문이 나옵니다. 실행 로그의 `Utilizing 9 out of 36 chunks`는 청크 36개 중 9개를 근거 묶음으로 썼다는 뜻입니다. 나머지 27개로는 질문을 만들지 않으므로, 골든셋이 문서 전체를 다루지는 않습니다. 어느 청크가 뽑힐지는 무작위라 실행마다 다른 주제의 질문이 나옵니다. 문서 전체를 고르게 다루려면 `max_contexts_per_document`를 늘리는데, 근거 묶음마다 LLM을 불러 시간과 비용이 함께 늘어납니다.
- `max_goldens_per_context=1`은 근거 묶음마다 골든 1개, `include_expected_output=True`는 정답 답변도 함께 만들라는 뜻입니다.
- 모델: 질문과 정답 답변은 1절의 생성 모델(`gpt-4.1`)이 만들고, 근거 묶음과 질문의 품질 검사(`critic_model`)는 판단 모델(`gpt-5.6-luna`)이 합니다. 만드는 모델과 검사하는 모델을 다르게 둡니다. 같은 모델이 자기가 만든 결과를 검사하면 자기 실수에 너그러워지기 쉽기 때문입니다.

생성 결과는 실행마다 달라집니다. 실무에서는 사람이 한 번 읽고 틀리거나 겹치는 골든을 고칩니다. 출력에서 볼 점은 이렇습니다.

- `근거(context)`는 그 질문을 만든 청크입니다. 정답 답변은 이 청크만 보고 만들어집니다.
- 청크 앞뒤에 깨진 글자(`�`)가 보이거나 문장이 중간에서 잘리는 것은, 한국어를 토큰 단위로 자르다 글자와 문장 경계가 끊긴 것입니다.
- 그래서 청크에 일부 문장만 들어가면 정답 답변도 일부만 담깁니다. 예를 들어 주말 운영 시간만 든 청크로 "운영 시간"을 물으면 평일 시간이 빠진 답이 만들어집니다. 4-3에서 판사가 이런 골든을 잡아냅니다.
- 비슷한 질문이 겹치기도 합니다.

In [ ]:
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import (
    ContextConstructionConfig,
    EvolutionConfig,
    FiltrationConfig,
    StylingConfig,
)


# 골든 생성기: 한국어 회원 질문과 짧은 답변을 만들도록 설정
synthesizer = Synthesizer(
    model=generation_model_name,  # 질문과 정답 답변을 만드는 LLM
    async_mode=True,  # 근거 묶음마다 골든을 동시에 생성(기본값 True)
    max_concurrent=32,  # 동시에 보낼 생성 요청 수 상한(기본값 100)
    styling_config=StylingConfig(
        scenario="헬스장 '핏플러스 강남점' 회원이 앱 채팅으로 이용 안내를 묻는다.",  # 헬스장 이름을 넣어야 다른 이름을 지어내지 않음
        task="헬스장 회원 안내 챗봇",
        input_format="회원이 쓰는 짧은 한국어 질문 한 문장",
        expected_output_format="안내 문서를 근거로 한 짧은 한국어 답변",
    ),
    evolution_config=EvolutionConfig(num_evolutions=0),  # 질문을 다시 써 어렵게 만드는 횟수(0: 그대로, 1: 한 번 고침(기본값), 2 이상: 그만큼 더 고침)
    filtration_config=FiltrationConfig(
        critic_model=judgment_model_name,  # 만든 질문의 품질 검사 LLM
        max_quality_retries=1,  # 질문 채점과 고쳐 쓰기 반복 횟수(기본값 3), 0이면 질문 품질 검사를 끔
    ),
)

# 안내 문서 파일을 직접 읽어 골든 생성
synthetic_goldens = await synthesizer.a_generate_goldens_from_docs(
    document_paths=[str(data_dir / "부록" / "fitplus_guide.md")],
    max_goldens_per_context=1,  # 근거 묶음마다 골든 1개
    include_expected_output=True,  # 정답 답변도 만듦
    context_construction_config=ContextConstructionConfig(
        embedder="text-embedding-3-small",  # 청크를 저장하고 비슷한 청크를 찾는 임베딩 모델
        critic_model=judgment_model_name,  # 근거 묶음의 품질 검사 LLM
        chunk_size=100,  # 청크 하나의 토큰 수
        chunk_overlap=0,  # 청크끼리 겹치는 토큰 수
        max_contexts_per_document=9,  # 문서 하나에서 만들 근거 묶음 수
        max_context_length=1,  # 근거 묶음 하나에 넣을 청크 수
        max_retries=1,  # 근거 묶음 수 x 이 값만큼 청크를 뽑아 채점하고 상위만 고름(기본값 3), 1이면 뽑은 청크를 모두 씀(채점 호출은 남음)
    ),
)

# 만든 골든 확인: 질문, 정답 답변, 그 질문을 만든 근거 청크와 출처 문서
for golden in synthetic_goldens:
    print("질문:", golden.input)
    print("정답 답변:", golden.expected_output)
    print("근거(context):", golden.context)
    print("출처 문서(source_file):", golden.source_file)
    print()

골든 생성은 LLM을 여러 번 불러 시간과 비용이 듭니다. 만든 골든을 파일로 저장해 두면, 다음부터는 위 셀을 다시 실행하지 않고 3-3에서 이 파일을 불러와 이어 갈 수 있습니다.

In [ ]:
from deepeval.dataset import EvaluationDataset

# 만든 골든을 JSON 파일로 저장(질문, 정답 답변, 근거, 출처 문서가 함께 저장됨)
saved_path = EvaluationDataset(goldens=synthetic_goldens).save_as(
    file_type="json",
    directory=str(data_dir / "부록"),
    file_name="synthetic_goldens",
)

### 3-3. 문서에 없는 질문은 직접 추가합니다

3-2의 골든 생성 셀을 건너뛰었다면 여기서 시작합니다(1절과 2절은 먼저 실행해 둡니다). 저장해 둔 합성 골든을 불러옵니다.

In [ ]:
from deepeval.dataset import EvaluationDataset

# 3-2에서 저장한 합성 골든 불러오기
saved_dataset = EvaluationDataset()
saved_dataset.add_goldens_from_json_file(str(data_dir / "부록" / "synthetic_goldens.json"))
synthetic_goldens = saved_dataset.goldens
print("불러온 합성 골든:", len(synthetic_goldens), "개")

`Synthesizer`는 근거 문장으로 답할 수 있는 질문만 만듭니다. 그런데 이 챗봇이 틀리는 것은 **문서에 답이 없는 질문**입니다. 이런 질문은 사람이 실제 문의 기록 등을 보고 직접 넣습니다. 정답 답변은 "문서에 없으니 프런트에 문의해 달라"는 안내입니다. 정답 근거(`context`)에는 "안내 문서에 이 질문에 대한 내용이 없음"이라고 적어, 근거를 빠뜨린 골든과 구분합니다. 판사는 `context`가 아니라 챗봇이 검색한 문장(`retrieval_context`)으로 채점하므로 채점에는 영향이 없습니다.

In [ ]:
from deepeval.dataset import Golden

# 문서에 없는 질문 4개와 공통 정답 답변
out_of_doc_questions = [
    "다른 지점에서도 운동할 수 있어요?",
    "일일 이용권 있어요?",
    "인바디 측정은 얼마예요?",
    "반려견을 데려가도 되나요?",
]
out_of_doc_answer = "안내 문서에 없는 내용이라 정확히 안내드리기 어려워요. 프런트에 문의해 주세요."

# 문서 밖 질문 골든: 근거가 없다는 것을 context에 적어 둡니다(빠뜨린 것과 구분하려고).
out_of_doc_goldens = []
for question in out_of_doc_questions:
    out_of_doc_goldens.append(Golden(
        input=question,
        expected_output=out_of_doc_answer,
        context=["안내 문서에 이 질문에 대한 내용이 없음"],
        source_file="data/부록/fitplus_guide.md",
    ))

# 합성 골든은 Synthesizer가 붙인 근거 청크(context)를 그대로 씁니다.
doc_goldens = synthetic_goldens

# 골든셋 전체: 합성 골든 + 직접 추가한 골든
golden_set = doc_goldens + out_of_doc_goldens

golden_rows = []
for golden in golden_set:
    golden_rows.append({
        "input": golden.input,
        "expected_output": golden.expected_output,
        "context": golden.context,
        "source_file": golden.source_file,
    })

print("문서에 있는 질문(합성)", len(doc_goldens), "개, 문서에 없는 질문(직접)", len(out_of_doc_goldens), "개")
display(pd.DataFrame(golden_rows))

## 4. Jev 판사 만들기

### 4-1. Jev와 판정 문장

Jev는 글을 쓰지 않는 판정 모델입니다. 판정 문장(`Noul`)을 주면 그 문장이 참일 확률을 돌려줍니다. 문장에는 `input`, `actual_output`, `retrieval_context` 같은 필드 이름을 그대로 쓰고, 한 문장에 한 가지만 묻습니다.

RAG 챗봇의 판사는 **챗봇이 실제로 검색해 본 문장(`retrieval_context`)** 을 기준으로 봅니다. 골든의 `context`(정답 근거)와 다릅니다. 검색기가 필요한 문장을 못 찾았는데 챗봇이 그럴듯하게 답했다면, 그 답은 본 근거에 없는 말을 한 것이기 때문입니다.

`strict_mode=True`로 두면 모든 문장의 참일 확률이 0.5 이상일 때만 pass입니다.

### 4-2. 판사 만들기와 첫 판정

판정 문장은 두 개입니다.

1. 검색된 문장에 답이 있으면 빠뜨리지 않고 말하고, 없을 때만 "없다"고 안내했는가
2. 검색된 문장에 없는 내용을 지어내지 않았는가

첫 문장을 "질문에 모두 답했는가"로만 쓰면, 문서에 없어서 정중히 거절한 올바른 답까지 떨어뜨립니다. 그래서 "답이 없을 때만 없다고 안내한다"를 함께 넣었습니다.

In [ ]:
from deepeval.metrics import JevEval
from deepeval.metrics.jev_eval import Noul
from deepeval.test_case import SingleTurnParams

accuracy_judge = JevEval(
    name="안내 정확성",
    # 판정 문장에서 가리키는 테스트 케이스 필드
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT],
    questions=[
        # 1. 검색된 문장에 답이 있으면 빠뜨리지 않고, 없을 때만 없다고 했는가
        Noul(
            "retrieval_context에 input의 답이 있으면 actual_output은 그 답을 빠뜨리지 않고 말한다. "
            "retrieval_context에 답이 없을 때만 없다고 안내한다."
        ),
        # 2. 검색된 문장에 없는 내용을 지어내지 않았는가
        Noul("actual_output은 retrieval_context에 있는 내용만 말하고, retrieval_context에 없는 시각·계획·약속·공식 확정을 지어내지 않는다."),
    ],
    system_one_model=jev_model_name,
    strict_mode=True,  # 두 문장이 모두 참이어야 pass
    async_mode=True,  # Jev 호출을 비동기로 보냄(DeepEval 기본값)
)

판사를 만들었으니 2-3에서 챗봇이 한 답을 판정해 봅니다. 그때 검색기가 찾은 문장을 `retrieval_context`로 함께 넣습니다.

In [ ]:
from deepeval.test_case import LLMTestCase

# 2-3의 답과, 그때 검색기가 찾은 문장을 테스트 케이스로 만들어 판정
demo_question = "PT 체험 수업은 무료예요?"
demo_found = []
for doc in retriever.invoke(demo_question):
    demo_found.append(doc.page_content)

demo_case = LLMTestCase(input=demo_question, actual_output=reply, retrieval_context=demo_found)
await accuracy_judge.a_measure(demo_case)  # 주피터에서는 비동기 버전을 await로 부릅니다

# 점수·통과 여부와 문장별 판정 이유
print("점수:", accuracy_judge.score, "/ 통과:", accuracy_judge.success)
print()
print(accuracy_judge.reason)

`reason`에는 판정 문장마다 참일 확률 `P(yes)`와 통과 여부(`strict=pass`·`strict=fail`)가 나옵니다. 답에 "보통 몇만 원" 같은 문서 밖 정보가 섞이면, 첫 문장은 통과해도 둘째 문장(지어내지 않는다)의 확률이 크게 낮아져 fail이 됩니다. 시험 실행에서는 매번 0.1 아래였습니다.

### 4-3. 골든셋으로 판사 점검: 정답 답변은 통과하는가

판사를 최적화에 맡기기 전에, 3절 골든셋의 정답 답변 13개를 판사에 넣어 봅니다. 사람이 정한(또는 검토한) 정답이므로 모두 pass여야 합니다. fail이 나오면 판사가 맞는 답을 떨어뜨린 것(오탐)이거나, 검색기가 정답에 필요한 문장을 찾지 못한 것입니다. 판사가 틀린 답을 통과시키거나 맞는 답을 떨어뜨리면 최적화도 그 방향으로 갑니다.

먼저 골든마다 검색기가 찾는 문장 4개를 `retrieval_context`에 붙입니다. 챗봇이 그 질문에 답할 때 실제로 보는 근거이고, 판사는 이 문장을 기준으로 채점합니다. 5절의 최적화와 전후 비교도 이 검색 결과를 씁니다.

지어낸 답을 fail로 잡는지는 4-2에서 봤고, 5-4에서 처음 프롬프트의 답으로 다시 확인합니다.

In [ ]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig

# 골든마다 챗봇이 실제로 보게 될 검색 결과(문장 4개)를 붙입니다.
for golden in golden_set:
    found_texts = []
    for doc in retriever.invoke(golden.input):
        found_texts.append(doc.page_content)
    golden.retrieval_context = found_texts

# 정답 답변을 챗봇 답변 자리에 넣은 테스트 케이스. name에는 골든 번호를 넣어 결과를 다시 붙입니다.
reference_cases = []
for number, golden in enumerate(golden_set):
    reference_cases.append(LLMTestCase(
        input=golden.input,
        actual_output=golden.expected_output,
        retrieval_context=golden.retrieval_context,
        name=str(number),
    ))

# 교안 02처럼 evaluate()로 한 번에 채점합니다.
reference_result = evaluate(
    test_cases=reference_cases,
    metrics=[accuracy_judge],
    async_config=AsyncConfig(max_concurrent=32),  # 동시에 보낼 케이스 수 상한
    display_config=DisplayConfig(print_results=False, show_indicator=False),  # 긴 기본 출력을 끕니다.
)

# 결과는 끝난 순서대로 오므로 name(골든 번호)으로 골든에 붙입니다.
reference_rows = []
for test_result in reference_result.test_results:
    golden = golden_set[int(test_result.name)]
    metric = test_result.metrics_data[0]
    if metric.success:
        label = "pass"
    else:
        label = "fail"
    reference_rows.append({
        "번호": int(test_result.name),
        "질문": golden.input,
        "정답 답변": golden.expected_output,
        "Jev": label,
        "Jev 이유": metric.reason,
    })

# 골든 순서대로 정렬해 확인
reference_table = pd.DataFrame(reference_rows).sort_values("번호", ignore_index=True)
print("정답 답변 pass:", (reference_table["Jev"] == "pass").sum(), "/", len(reference_table))
display(reference_table)

In [ ]:
# fail이 나온 골든만 골라, 골든을 만든 근거(context)와 판사가 본 검색 결과(retrieval_context)를 함께 봅니다.
fail_table = reference_table[reference_table["Jev"] == "fail"]
for _, row in fail_table.iterrows():
    golden = golden_set[row["번호"]]
    print(f"[골든 {row['번호']}] 질문:", golden.input)
    print("정답 답변:", golden.expected_output)
    print("골든을 만든 근거(context):", golden.context)
    print("판사가 본 검색 결과(retrieval_context):")
    for text in golden.retrieval_context:
        print("  -", text)
    print("Jev 이유:", row["Jev 이유"])
    print()

`evaluate()` 끝에 나오는 하이퍼파라미터 경고와 Confident AI 안내는 이 실습과 관계없습니다.

여러 번 시험 실행해 보면 13개 중 9~11개가 pass였고, 문서에 없는 질문의 거절 답은 매번 통과했습니다. fail이 나온 골든은 위 셀에서 정답 답변, 골든을 만든 근거, 검색 결과, `Jev 이유`를 함께 읽어 봅니다. 시험 실행에서 fail은 대부분 `Synthesizer`가 잘못 만든 골든이었고, 원인은 주로 3-2에서 본 청크 경계였습니다.

- 청크에 일부 문장만 들어가 정답 답변도 일부만 담긴 경우: 운영 시간에 주말 시간만, 회원권 가격에 일부 가격만 적혔습니다.
- 청크가 문장 중간에서 시작해 핵심 정보가 빠진 경우: 위치를 묻는데 층수가 빠졌습니다.
- 정답 답변이 문서와 다르게 쓰인 경우: 환불을 "남은 기간 금액에서"가 아니라 "전체 금액에서" 위약금을 뺀다고 적었습니다.

이런 fail은 판사가 맞는 답을 떨어뜨린 것이 아니라 틀린 골든을 잡아낸 것입니다. 정답 답변이 맞는데 fail이면, 검색기가 정답에 필요한 문장을 4개 안에 다 찾지 못한 경우입니다. 이때는 검색기 설정(`k`)이나 문서 나누기를 먼저 봅니다. 실무에서는 이런 골든을 고친 뒤 최적화에 씁니다. 이 부록은 사람 검토가 왜 필요한지 보이려고 합성 결과를 그대로 둡니다. 판사는 정답 답변을 보지 않고 검색된 문장으로 채점하므로, 5-4의 전후 비교에는 영향이 없습니다.

## 5. Prompt Optimization: 판사 점수로 프롬프트 고치기

### 5-1. PromptOptimizer와 GEPA

- `Prompt`: 고칠 시스템 프롬프트. 휴먼 프롬프트는 그대로 둡니다.
- 골든: 최적화용 골든 4개
- `metrics`: 4절의 판사
- `model_callback`: 후보 프롬프트와 골든을 받아 답을 돌려주는 함수. `async def`로 쓰면 GEPA가 한 회차 안의 여러 골든을 동시에 처리합니다. 회차끼리는 앞 회차 결과로 다음 후보를 만들기 때문에 차례로 돕니다.

프롬프트를 고치는 방법(알고리즘)은 `algorithm=`으로 고르며, 기본값은 GEPA입니다. 답변 모델·수정 LLM·Jev를 여러 번 불러 몇 분 걸립니다.

**GEPA**(Genetic-Pareto)는 DSPy 연구에서 나온 방법을 DeepEval이 옮긴 것입니다. 이름의 두 단어가 하는 일을 나타냅니다.

- **Genetic(진화):** 부모 프롬프트를 고쳐 자식 프롬프트를 만들고, 좋은 자식만 남겨 다음 회차의 부모로 씁니다. 고칠 때는 수정 LLM이 틀린 답과 판사의 이유를 읽고 무엇이 문제인지 진단한 뒤 새 문장을 씁니다.
- **Pareto(파레토):** 가장 좋은 프롬프트 하나만 남기지 않고, 골든마다 강점이 다른 후보를 함께 남깁니다. 다른 후보에게 완전히 밀리는 후보만 버립니다. 완전히 밀린다는 것은 어느 골든에서도 더 낫지 않고, 한 골든 이상에서 더 못하다는 뜻입니다.

GEPA는 최적화용 골든을 **고칠 점 찾기용**과 **후보 비교용**으로 나눠 쓰고, 회차마다 이렇게 돕니다.

1. 남아 있는 후보 중 부모를 하나 고릅니다.
2. 고칠 점 찾기용 골든으로 답을 만들어 채점하고, 수정 LLM이 프롬프트를 고칩니다.
3. 고친 프롬프트가 같은 골든에서 점수를 올리지 못하면 버립니다(`skipped`).
4. 후보 비교용 골든에서 기존 후보들에게 밀리면 버리고(`rejected`), 아니면 남깁니다(`accepted`).
5. 모든 회차가 끝나면 비교용 평균 점수가 가장 높은 후보를 고르고, 같으면 나중에 고친 후보를 고릅니다.

참고로 DeepEval에는 GEPA 말고도 세 가지 알고리즘이 있고, `algorithm=`에 넣어 바꿀 수 있습니다. 모두 "후보 프롬프트를 만들어 판사로 채점하고 좋은 쪽을 남긴다"는 틀은 같고, 후보를 만드는 방식이 다릅니다.

| 알고리즘 | 후보를 만드는 방식 | 맞는 경우 |
|---|---|---|
| GEPA(이 부록) | 틀린 답을 진단해 고치고, 강점이 다른 후보를 함께 남김 | 질문 유형마다 잘 맞는 프롬프트가 다를 때 |
| MIPROv2 | 지시문 후보와 예시(few-shot) 묶음의 조합을 골라 시험 | 예시를 보여 주면 답이 크게 좋아질 때 |
| COPRO | 지시문 후보 여러 개 중 가장 좋은 것을 기준으로 단계마다 다시 씀 | 지시문만 다듬으면 될 때 |
| SIMBA | 같은 질문에 여러 번 답하게 해, 답이 흔들리는 질문의 잘된 답을 예시·규칙으로 넣음 | 같은 질문에서 성공과 실패가 섞여 나올 때 |

이 부록의 챗봇은 질문 유형(문서에 있는 질문, 없는 질문)에 따라 결과가 갈리므로 GEPA를 씁니다. 자세한 설명은 [GEPA](https://deepeval.com/docs/prompt-optimization-gepa) · [MIPROv2](https://deepeval.com/docs/prompt-optimization-miprov2) · [COPRO](https://deepeval.com/docs/prompt-optimization-copro) · [SIMBA](https://deepeval.com/docs/prompt-optimization-simba) 문서에 있습니다.

### 5-2. 최적화 실행

**골든 나누기:** 골든셋은 둘로 나눕니다.

- **최적화용:** 연습 문제. 최적화가 이 질문들의 점수를 보며 프롬프트를 고칩니다. 문서에 없는 질문 2개와 합성 골든 2개입니다.
- **확인용:** 실제 시험. 나머지(합성 골든 7개, 문서에 없는 질문 2개)로, 최적화가 끝날 때까지 쓰지 않습니다. 고친 프롬프트가 연습 문제에만 맞춰진 것이 아닌지 이 골든으로 확인합니다.

GEPA는 시작할 때 최적화용 골든 4개를 `random_seed`로 한 번 두 묶음으로 나누고, 끝까지 다시 나누지 않습니다. `random_seed=42`와 아래 목록 순서에서는 이렇게 나뉩니다.

| 묶음 | 골든 | 하는 일 | 설정 |
|---|---|---|---|
| 고칠 점 찾기용 | 문서에 없는 질문 2개(다른 지점, 일일 이용권). 시작할 때 정해져 끝까지 고정 | 회차마다 이 묶음에서 골든을 뽑아 현재 프롬프트로 답을 만들어 채점하고, 점수·이유·정답 답변을 수정 LLM에게 보여 줘 프롬프트를 고쳐 쓰게 합니다. 고친 프롬프트로 다시 채점해 점수가 오르지 않으면 버립니다(`skipped`). | `minibatch_size=2`: 회차마다 이 묶음에서 무작위로 뽑는 골든 수(같은 골든이 두 번 뽑힐 수 있음) |
| 후보 비교용 | 3-2에서 만든 합성 골든 2개(문서에 있는 질문). 시작할 때 정해져 끝까지 고정 | 점수가 오른 후보를 이 골든으로 다시 채점해, 기존 후보보다 못하면 버리고(`rejected`) 아니면 남깁니다(`accepted`). 최종 프롬프트도 이 점수로 고릅니다. | `pareto_size=2`: 최적화용 골든에서 무작위로 떼어 두는 골든 수 |

후보 비교용은 최적화용 골든 안에서 떼어 둔 것이고, 확인용 골든과는 다릅니다.

고칠 점 찾기용을 처음 프롬프트가 이미 다 맞히면 고칠 이유가 없어 처음 프롬프트가 그대로 돌아옵니다. 그래서 지금 챗봇이 틀리는 질문이 이쪽에 들어가도록 골든 순서를 정했습니다.

In [ ]:
# 최적화용: 문서에 없는 질문 2개 + 합성 골든 2개
train_goldens = [out_of_doc_goldens[0], doc_goldens[0], doc_goldens[1], out_of_doc_goldens[1]]
# 확인용: 나머지 전부
check_goldens = doc_goldens[2:] + out_of_doc_goldens[2:]

# 용도별 골든을 표로 확인
split_rows = []
for name, goldens in [("최적화", train_goldens), ("확인", check_goldens)]:
    for golden in goldens:
        split_rows.append({"용도": name, "input": golden.input})

print("최적화용", len(train_goldens), "개, 확인용", len(check_goldens), "개")
display(pd.DataFrame(split_rows))

**라이브러리 버그 우회:** DeepEval 4.2.5부터 최신 4.2.9까지의 `PromptOptimizer`는 OpenAI 모델을 쓰면 `AttributeError: 'Scorer' object has no attribute '_accrue_tokens'`로 멈춥니다. 빠진 메서드를 빈 메서드로 채워 우회합니다. 버그가 고쳐진 버전에서는 필요 없습니다.

In [ ]:
from deepeval.optimizer.rewriter import Rewriter
from deepeval.optimizer.scorer import Scorer


# 빠진 토큰 기록 메서드를 빈 메서드로 채워 넣습니다(DeepEval 4.2.5~4.2.9 버그 우회).
def skip_token_count(self, input_tokens, output_tokens):
    # 토큰 수를 기록하지 않고 넘어갑니다.
    pass


Scorer._accrue_tokens = skip_token_count
Rewriter._accrue_tokens = skip_token_count

**최적화 실행:** `answer_with_prompt`가 후보 시스템 프롬프트로 2-3과 같은 RAG 체인을 만들어 답합니다.

In [ ]:
from deepeval.optimizer import PromptOptimizer
from deepeval.optimizer.algorithms import GEPA
from deepeval.prompt import Prompt


tried_prompts = []  # 최적화가 시험한 후보 시스템 프롬프트를 나온 순서대로 모읍니다.


async def answer_with_prompt(prompt, golden):
    # 처음 보는 후보면 기록해 둡니다(5-3에서 최적화 과정을 볼 때 씁니다).
    if prompt.text_template not in tried_prompts:
        tried_prompts.append(prompt.text_template)

    # 후보 시스템 프롬프트로 2-3과 같은 RAG 체인을 만듭니다. 검색기와 휴먼 프롬프트는 그대로입니다.
    candidate_prompt = ChatPromptTemplate.from_messages([
        ("system", prompt.text_template),
        ("human", "[안내 문서]\n{context}\n\n[회원 질문]\n{input}"),
    ])
    candidate_chain = (
        {"context": retriever | format_docs, "input": RunnablePassthrough()}
        | candidate_prompt
        | answer_llm
        | StrOutputParser()
    )
    # 비동기(ainvoke)로 답하면 GEPA가 여러 골든의 답을 동시에 만들고 채점합니다.
    return await candidate_chain.ainvoke(golden.input)


# 고칠 대상: 2-3의 시스템 프롬프트
initial_prompt = Prompt(text_template=initial_system_prompt)

# 최적화 설정: 답변 함수, 판사, 수정 LLM, 알고리즘
optimizer = PromptOptimizer(
    model_callback=answer_with_prompt,
    metrics=[accuracy_judge],
    optimizer_model=judgment_model_name,  # 프롬프트를 진단하고 고쳐 쓰는 LLM(1절의 판단 모델)
    algorithm=GEPA(
        iterations=2,  # 프롬프트를 고쳐 보는 횟수
        pareto_size=2,  # 후보 비교용으로 떼어 둘 골든 수(합성 골든 2개)
        minibatch_size=2,  # 회차마다 고칠 점 찾기용(다른 지점, 일일 이용권)에서 뽑는 골든 수
        random_seed=42,  # 골든 나누기와 뽑기를 매번 같게
    ),
)
# 최적화용 골든으로 최적화 실행(주피터에서는 비동기 버전을 await로 부릅니다)
optimized_prompt = await optimizer.a_optimize(prompt=initial_prompt, goldens=train_goldens)

# 최종으로 고른 시스템 프롬프트
print(optimized_prompt.text_template)

표의 `accepted`는 채택, `skipped`·`rejected`는 버린 후보이고, `★`가 최종 프롬프트입니다. `Before`·`After`는 회차마다 기준 골든이 달라 서로 비교하지 않습니다.

최종 프롬프트에서 처음의 실수 문장이 어떻게 바뀌었는지 봅니다. 회차별 결과(`accepted`, `skipped`, `rejected`, 고친 문장이 부모와 같아 건너뛴 `child == parent`)는 실행마다 달랐지만, 시험 실행에서는 매번 최종 프롬프트가 실수 문장을 지우고, "문서에 없는 내용은 추측하지 말고 프런트에 문의하라고 안내해" 같은 규칙을 넣었습니다(문장은 매번 달랐습니다).

### 5-3. 최적화 과정 보기

최적화가 시험한 후보 프롬프트를 나온 순서대로 봅니다. `optimizer.optimization_report`의 `pareto_scores`에는 끝까지 남은 후보와 비교용 골든 점수가 들어 있습니다. 처음에 채택됐더라도 나중에 더 나은 후보에게 완전히 밀리면 여기서 빠집니다.

In [ ]:
# 최적화 리포트: 후보로 남은 프롬프트와 비교용 점수가 들어 있습니다.
report = optimizer.optimization_report

# 끝까지 남은 후보(pareto_scores에 있는 것) -> 비교용 골든 점수
kept_scores = {}
for config_id, scores in report.pareto_scores.items():
    for kept_prompt in report.prompt_configurations[config_id].prompts.values():
        kept_scores[kept_prompt.text_template] = scores

# 시험한 후보마다 결과(최종 선택, 끝까지 남음, 버림) 표시
process_rows = []
for number, prompt_text in enumerate(tried_prompts):
    if prompt_text == optimized_prompt.text_template:
        result = "최종 선택"
    elif prompt_text in kept_scores:
        result = "끝까지 남음"
    else:
        result = "버림(또는 더 나은 후보에게 밀려 빠짐)"
    if number == 0:
        result = "처음 프롬프트, " + result
    process_rows.append({
        "후보": number,
        "결과": result,
        "비교용 점수": kept_scores.get(prompt_text, "-"),
        "시스템 프롬프트": prompt_text,
    })

display(pd.DataFrame(process_rows))

표를 읽는 법은 이렇습니다.

- `결과`가 "최종 선택"인 후보가 5-2에서 출력한 최종 프롬프트입니다. 처음 프롬프트가 "버림"이면, 고친 후보에게 완전히 밀려 빠졌다는 뜻입니다.
- `비교용 점수`는 후보 비교용 골든 2개의 점수입니다. 예를 들어 `[1.0, 0.0]`이면 하나는 pass, 하나는 fail입니다.
- 위에서 아래로 후보 문장을 읽으면, 수정 LLM이 판사의 이유를 보고 문장을 어떻게 바꿔 갔는지 보입니다.

점수가 같은 후보가 여럿이면 나중에 고친 후보가 뽑히므로, 고른 프롬프트는 사람이 한 번 읽고 정합니다.

### 5-4. 확인용 골든으로 전후 비교

확인용 골든 9개에 처음 프롬프트와 최적화 프롬프트로 답하게 하고, 같은 판사로 채점합니다. 답 생성은 `batch`, 채점은 `evaluate()`로 동시에 보내 빠르게 끝냅니다.

In [ ]:
check_questions = []
for golden in check_goldens:
    check_questions.append(golden.input)

# 최적화 프롬프트 체인: 2-3의 rag_chain에서 시스템 프롬프트만 바꿉니다.
optimized_prompt_template = ChatPromptTemplate.from_messages([
    ("system", optimized_prompt.text_template),
    ("human", "[안내 문서]\n{context}\n\n[회원 질문]\n{input}"),
])
optimized_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | optimized_prompt_template
    | answer_llm
    | StrOutputParser()
)

# 확인용 질문에 답합니다. 처음 프롬프트는 2-3의 rag_chain을 그대로 씁니다.
# batch는 질문 여러 개를 동시에 보내고, 결과는 입력 순서대로 돌려줍니다.
replies_by_prompt = {
    "처음": rag_chain.batch(check_questions, config={"max_concurrency": 32}),
    "최적화": optimized_chain.batch(check_questions, config={"max_concurrency": 32}),
}

# 답 18개를 테스트 케이스로 만듭니다. name에는 "프롬프트 이름/골든 번호"를 넣어 결과를 다시 붙입니다.
compare_cases = []
for prompt_name, replies in replies_by_prompt.items():
    for number, golden in enumerate(check_goldens):
        compare_cases.append(LLMTestCase(
            input=golden.input,
            actual_output=replies[number],
            retrieval_context=golden.retrieval_context,
            name=prompt_name + "/" + str(number),
        ))

# evaluate()로 18건을 한 번에 채점합니다.
compare_result = evaluate(
    test_cases=compare_cases,
    metrics=[accuracy_judge],
    async_config=AsyncConfig(max_concurrent=32),  # 동시에 보낼 케이스 수 상한
    display_config=DisplayConfig(print_results=False, show_indicator=False),  # 긴 기본 출력을 끕니다.
)

# name -> pass/fail
label_by_name = {}
for test_result in compare_result.test_results:
    if test_result.metrics_data[0].success:
        label_by_name[test_result.name] = "pass"
    else:
        label_by_name[test_result.name] = "fail"

# 질문마다 처음·최적화 답변과 판정을 한 행으로 모읍니다.
rows = []
for number, golden in enumerate(check_goldens):
    if golden in out_of_doc_goldens:
        question_type = "문서에 없음"
    else:
        question_type = "문서에 있음"
    rows.append({
        "유형": question_type,
        "질문": golden.input,
        "처음 답변": replies_by_prompt["처음"][number],
        "처음 판정": label_by_name["처음/" + str(number)],
        "최적화 답변": replies_by_prompt["최적화"][number],
        "최적화 판정": label_by_name["최적화/" + str(number)],
    })

compare_table = pd.DataFrame(rows)
display(compare_table)

In [ ]:
# 통과율(pass 비율): 질문 유형별과 전체
pass_check = pd.DataFrame({
    "유형": compare_table["유형"],
    "처음": compare_table["처음 판정"] == "pass",
    "최적화": compare_table["최적화 판정"] == "pass",
})
pass_rate = pass_check.groupby("유형")[["처음", "최적화"]].mean()
pass_rate.loc["전체"] = pass_check[["처음", "최적화"]].mean()
display(pass_rate.style.format("{:.0%}"))

통과율은 문서에 있는 질문(답을 빠뜨리지 않는가)과 없는 질문(지어내지 않는가)을 나눠 봅니다. 전체만 보면 한쪽이 나빠진 것을 놓칩니다. 여러 번 시험 실행해 보면 전체 통과율은 처음 프롬프트 44~67%(4~6/9), 최적화 프롬프트 89~100%(8~9/9)였습니다.

- 처음 프롬프트는 문서에 없는 질문(인바디, 반려견)에서 매번 "보통 회원은 무료" 같은 내용을 지어냈습니다. 문서에 있는 질문에서도 문서에 없는 절차를 덧붙여 떨어지곤 했습니다. "흔한 내용으로 구체적으로 답해"라는 실수 문장이 답을 부풀린 것입니다.
- 최적화 프롬프트가 떨어진 경우는 모두, 문서에 답이 있는 질문(일시 정지, 회원권 가격)까지 "문서에 없다"고 거절한 경우였습니다. 규칙이 너무 강해지면 이런 과잉 거절이 생깁니다. 그래서 확인용 골든에는 문서에 있는 질문을 꼭 함께 둡니다.

LLM이 실행마다 다른 글을 쓰므로, 다시 실행하면 고친 프롬프트와 결과가 달라집니다. 실무에서는 확인용 골든을 더 늘리고 고친 프롬프트를 사람이 한 번 읽은 뒤 배포합니다.